In [1]:
import torch
from scope_diffuser import SCoPEDiffusionPipeline

import matplotlib.pyplot as plt
import random
import numpy as np
step_size = 28
for temperature in [0.85, 0.9, 1, 5, 10, 20]:
    torch.manual_seed(42)
    # Load the model
    model_id = "CompVis/stable-diffusion-v1-4"
    pipe = SCoPEDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True)
    pipe = pipe.to("cuda:1")

    # Initialize list to store intermediate images
    intermediate_images = []

    # Callback function to save intermediate steps
    def save_intermediate(step, timestep, latents):
        with torch.no_grad():
            latents = 1 / 0.18215 * latents
            image = pipe.vae.decode(latents).sample
            image = (image / 2 + 0.5).clamp(0, 1)
            image = image.cpu().permute(0, 2, 3, 1).float().numpy()[0]
            intermediate_images.append(image)

    # for scope diffusion

    # prompt_schedule = [
    #     (0, "A clear night sky with a bright full moon and twinkling stars"),  
    #     (step_size, "A clear night sky with a bright full moon over a simple city skyline with minimal lighting"),
    #     (step_size * 2, "A clear night sky with a bright full moon over a vibrant city skyline with tall illuminated buildings and a serene river"),
    #     (step_size * 3, "A clear night sky with a bright full moon over a vibrant city skyline with tall illuminated buildings and a serene river, colorful lights, and reflections on the water"),
    #     (step_size * 4, "A clear starry night sky with a bright full moon over a vibrant futuristic city skyline with tall illuminated buildings and a serene river, colorful neon lights, flying cars, and reflections on the water ")
    # ]
    prompt_schedule = [
        (0, "A marketplace at night"),  
        (step_size, "An Indian marketplace at night with small shops"),
        (step_size * 2, "An Indian marketplace at night with small shops, and people walking around in groups"),
        (step_size * 3, "An Indian marketplace at night with small shops selling vegetables, and people walking around in groups "),
        (step_size * 4, "An Indian marketplace at night under a full moon with small shops selling vegetables, and people walking around in groups taking on phones")
    ]

    # prompt_schedule = [
    #     (0, "A bustling city street at night"),
    #     (step_size, "A bustling city street at night, rain falling lightly, reflecting neon lights from the signs above"),
    #     (step_size * 2, "A bustling city street at night, rain falling lightly, neon signs reflecting on the wet pavement, people huddled under umbrellas"),
    #     (step_size * 3, "A bustling city street at night, rain falling lightly, neon signs reflecting on the wet pavement, people huddled under umbrellas, a street vendor selling hot snacks under a small canopy"),
    #     (step_size * 4, "A bustling city street at night, rain falling lightly, neon signs reflecting on the wet pavement, people huddled under umbrellas, a street vendor selling hot snacks under a small canopy, a couple walking closely, sharing an umbrella")
    # ]


    num_inference_steps = 200

    image = pipe(
        prompt_schedule,
        temperature = temperature,
        num_inference_steps=num_inference_steps,
        callback=save_intermediate,
        callback_steps=1
    ).images[0]

    # Convert PIL Image to numpy array
    image_np = np.array(image)

    # Plot random samples
    num_samples = 5
    random_indices = [step_size, step_size*2, step_size*3, step_size*4, step_size*5]
    random_indices.sort()

    plt.figure(figsize=(20, 4))
    for i, idx in enumerate(random_indices):
        plt.subplot(1, num_samples +1, i + 1)
        plt.imshow(intermediate_images[idx])
        plt.title(f"After prompt: {i} Step {idx}")
        plt.axis('off')

    # Plot final image
    plt.subplot(1, num_samples+1, num_samples+1)
    plt.imshow(image_np)
    plt.title("Final Image")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

2024-10-05 10:36:24.085517: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-05 10:36:24.765518: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


10:36:25 __init__.py:40 [I] → Notebook logger initialized.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/ketanss/anaconda3/envs/opl_1/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/ketanss/files/scope/scope-diffusers/scope_diffuser/pipeline_stable_diffusion.py:303: FutureWarning: `callback` is deprecated and will be removed in version 1.0.0. Passing `callback` as an input argument to `__call__` is deprecated, consider using `callback_on_step_end`
  deprecate(
/home/ketanss/files/scope/scope-diffusers/scope_diffuser/pipeline_stable_diffusion.py:309: FutureWarning: `callback_steps` is deprecated and will be removed in version 1.0.0. Passing `callback_steps` as an input argument to `__call__` is deprecated, consider using `callback_on_step_end`
  depreca

  0%|          | 0/200 [00:00<?, ?it/s]